In [1]:
import pandas as pd
import numpy as np
import glob as gb,os
import re, time, json, os, shutil
import requests
import csv
from pathlib import Path
from getpass import getpass
%pip install lyricsgenius
%pip install syncedlyrics
from lyricsgenius import Genius
import syncedlyrics

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
countrys = ["us", "jp", "gb", "de", "fr", "ca", "au", "br", "mx"]
## sem china e south coreia (motivos obvios)

years = [2018,2019,2020,2021,2022,2023,2024,2025]

df = pd.concat(
    (pd.read_csv(f) for f in gb.glob("./Datasets/*csv")),
    ignore_index=True
)

df = df[df["cc"].isin(countrys)]
df["week_end"] = pd.to_datetime(df["week_end"])
df["year"] = df["week_end"].dt.year
df = df[df["year"].isin(years)]
df["pontos"] = 201 - df["rank"]


In [3]:
dfs = []
for c in countrys:
    caminho = f"./Datasets/MGDplus/charts/regional/spotify_charts_regional_{c}.csv"
    d = pd.read_csv(caminho, sep="\t", thousands=",")
    d["arquivo"] = os.path.basename(caminho)
    dfs.append(d)

other_df = pd.concat(dfs, ignore_index=True)

In [4]:
d = other_df.rename(columns={
    "market": "cc",
    "ID": "track_id",
    "Track": "title",
    "Artist": "artists",
    "Streams": "streams",
    "Date": "date",
    "Position": "rank_diario",
})

d = d[["cc", "date", "track_id", "title", "artists", "streams"]].copy()
d["date"] = pd.to_datetime(d["date"])
d["cc"] = d["cc"].str.lower()

# 1. Rótulo da semana = quinta-feira final (semana sex-qui, igual ao Spotify)
d["week_end"] = d["date"] + pd.to_timedelta((3 - d["date"].dt.weekday) % 7, unit="D")

# 2. Mantém só semanas completas (7 dias distintos) por país
dias_semana = d.groupby(["cc", "week_end"])["date"].transform("nunique")
d = d[dias_semana == 7]

# 3. Soma diária -> semanal por música
semanal = (
    d.groupby(["cc", "week_end", "track_id"], as_index=False)
     .agg(streams=("streams", "sum"),
          title=("title", "first"),
          artists=("artists", "first"))
)

# 4. Recria o rank semanal
semanal["rank"] = (
    semanal.groupby(["cc", "week_end"])["streams"]
           .rank(method="first", ascending=False)
           .astype(int)
)

semanal = semanal[semanal["rank"] <= 200]          # mesmo tamanho do chart oficial
semanal["pontos"] = 201 - semanal["rank"]

semanal["year"] = semanal["week_end"].dt.year

semanal = semanal[semanal["year"] == 2017]

cols = ["cc", "week_end", "track_id", "title", "artists", "streams", "rank", "pontos", "year"]

final = pd.concat([df[cols], semanal[cols]], ignore_index=True)

agg = (
    final.groupby(["cc", "year", "track_id"], as_index=False).agg(
        pontos=("pontos", "sum"),
        rank_medio=("rank", "mean"),
        melhor_posicao=("rank", "min"),
        semanas=("week_end", "nunique"),
        streams=("streams", "sum"),
        title=("title", "first"),
        artists=("artists", "first"),
    )
)

top40 = (
    agg.sort_values(["cc", "year", "pontos"], ascending=[True, True, False])
       .groupby(["cc", "year"])
       .head(120)
)

/tmp/ipykernel_162069/1923356558.py:16: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  d["week_end"] = d["date"] + pd.to_timedelta((3 - d["date"].dt.weekday) % 7, unit="D")


In [5]:
print(top40.groupby(["cc", "year"]).size().unstack())

year  2017  2018  2019  2020  2021  2022  2023  2024  2025
cc                                                        
au     120   120   120   120   120   120   120   120   120
br     120   120   120   120   120   120   120   120   120
ca     120   120   120   120   120   120   120   120   120
de     120   120   120   120   120   120   120   120   120
fr     120   120   120   120   120   120   120   120   120
gb     120   120   120   120   120   120   120   120   120
jp     120   120   120   120   120   120   120   120   120
mx     120   120   120   120   120   120   120   120   120
us     120   120   120   120   120   120   120   120   120


In [6]:
_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u2028\u2029]")

def _limpa_texto(s, max_chars, proteger_formulas):
    if s is None or pd.isna(s):
        return ""
    s = str(s)
    s = s.encode("utf-8", errors="replace").decode("utf-8")   # remove surrogates inválidos
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _CTRL.sub("", s).strip()
    if max_chars and len(s) > max_chars:
        s = s[:max_chars]
    if proteger_formulas and s[:1] in ("=", "+", "-", "@"):
        s = " " + s
    return s

def saveArchiveB(data, caminho, sep=",", max_chars=32000,
                 proteger_formulas=True, salvar_parquet=False):
    """Salva CSV seguro para Excel/Python. Não altera o DataFrame original."""
    out = data.copy()

    # remove colunas de sobra de merges anteriores
    out = out.drop(columns=[c for c in out.columns
                            if c == "Unnamed: 0" or str(c).endswith(("_x", "_y"))])

    # limpa todas as colunas de texto (letra, title, artists, ...)
    cols_txt = out.select_dtypes(include=["object", "string"]).columns
    truncadas = 0
    for c in cols_txt:
        if c == "letra":
            truncadas = (out[c].fillna("").astype(str).str.len() > max_chars).sum()
        out[c] = out[c].map(lambda s: _limpa_texto(s, max_chars, proteger_formulas))

    arquivo = caminho if caminho.lower().endswith(".csv") else caminho + ".csv"
    os.makedirs(os.path.dirname(arquivo) or ".", exist_ok=True)

    out.to_csv(
        arquivo,
        index=False,
        sep=sep,
        encoding="utf-8-sig",            # BOM: o Excel reconhece acentos, hangul e kana
        quoting=csv.QUOTE_NONNUMERIC,    # texto sempre entre aspas
        lineterminator="\r\n",
        errors="replace",
    )

    if salvar_parquet:                    # backup sem perda (precisa de pyarrow)
        out.to_parquet(arquivo.replace(".csv", ".parquet"), index=False)

    # verificação: reler e comparar
    chk = pd.read_csv(arquivo, sep=sep, keep_default_na=False, dtype=str, encoding="utf-8-sig")
    ok_forma = chk.shape == out.shape
    ok_letra = (chk["letra"].reset_index(drop=True)
                == out["letra"].fillna("").astype(str).reset_index(drop=True)).all() \
               if "letra" in out.columns else True
    print(f"Salvo em {arquivo} | {out.shape[0]} linhas x {out.shape[1]} colunas")
    print(f"Forma preservada: {ok_forma} | Letras idênticas: {ok_letra} | Letras truncadas: {truncadas}")
    return arquivo

In [7]:
# ---------- TOKEN DO GENIUS ----------
# ⚠️ REVOGUE este token em https://genius.com/api-clients e mova para .env / getpass.
# Ele está exposto no histórico do git.
if not os.environ.get("GENIUS_ACCESS_TOKEN"):
    os.environ["GENIUS_ACCESS_TOKEN"] = "CjxE91ETk4YyaUWcRcVdQkRAsvkc8Ec3qUQVTaGzoffzQ99cAVbq-e-98NATAGVL"

# ---------- CONSTANTES ----------
LRCLIB_BASE    = "https://lrclib.net/api"
LYRICSOVH_BASE = "https://api.lyrics.ovh/v1"

HEADERS = {"User-Agent": "top40-letras/0.3 (seu-email-ou-repo)"}
session = requests.Session()
session.headers.update(HEADERS)

PASTA_DADOS = Path("./dados").resolve()
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
CACHE = PASTA_DADOS / "letras_cache.jsonl"
print("Cache em:", CACHE)

# ---------- HELPERS DE LIMPEZA ----------
PADROES_VERSAO = [
    # Inglês (US, GB, CA, AU)
    r"acoustic", r"live", r"remaster(?:ed)?", r"remix", r"version", r"edit",
    r"demo", r"instrumental", r"radio\s*edit", r"extended", r"deluxe",
    r"bonus", r"alternate", r"take", r"session", r"unplugged", r"stripped",
    # Português (BR)
    r"ao\s*vivo", r"acústico", r"remasterizado", r"versão", r"edição",
    r"demonstração", r"instrumental", r"Ao\s*Vivo",
    # Espanhol (MX)
    r"en\s*vivo", r"acústico", r"remasterizado", r"versión", r"edición",
    r"demostración", r"instrumental",
    # Japonês (JP)
    r"アコースティック", r"ライブ", r"リマスター", r"リミックス",
    r"バージョン", r"エディット", r"デモ", r"インストゥルメンタル",
    # Alemão (DE)
    r"akustisch", r"live", r"remaster(?:ed|t)?", r"remix", r"version",
    r"edit", r"demo", r"instrumental", r"radio\s*edit", r"extended",
    r"deluxe", r"bonus", r"alternativ(?:e)?", r"take", r"session",
    r"unplugged", r"stripped",
    # Francês (FR, CA)
    r"acoustique", r"live", r"remasteris(?:é|ée|és|ées)?", r"remix",
    r"version", r"edit", r"édition", r"démo", r"instrumental",
    r"radio\s*edit", r"extended", r"deluxe", r"bonus",
    r"alternatif|alternative", r"prise", r"session", r"unplugged",
    r"stripped", r"en\s*concert",
]

PADRAO_VERSAO_RE = re.compile(
    r"""
    \s*[\(\[]\s*(?:""" + "|".join(PADROES_VERSAO) + r""")[^\)\]]*\s*[\)\]]
    |
    \s*-\s*(?:""" + "|".join(PADROES_VERSAO) + r""").*$
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

def limpa_titulo(t: str) -> str:
    t = PADRAO_VERSAO_RE.sub("", t)
    t = re.sub(r"\s*-\s*Ao Vivo.*$", "", t, flags=re.I)
    t = re.sub(r"\s*\((?:feat|ft|with|part)\.?[^)]*\)", "", t, flags=re.I)
    t = re.sub(r"\s*\[(?:feat|ft|with|part)\.?[^\]]*\]", "", t, flags=re.I)
    t = re.sub(r"\s*-\s*$", "", t)
    return t.strip()

def primeiro_artista(a: str) -> str:
    return re.split(r"\s*(?:\||,|&|feat\.?|ft\.?)\s*", a, flags=re.I)[0].strip()

# ---------- API 1: LRCLIB ----------
def busca_lrclib(titulo: str, artista: str, duracao=None) -> dict | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    params = {"track_name": t, "artist_name": a}
    if duracao is not None and not pd.isna(duracao):
        params["duration"] = int(duracao)

    try:
        r = session.get(f"{LRCLIB_BASE}/get", params=params, timeout=15)
        if r.status_code == 200:
            return r.json()
    except requests.RequestException:
        pass

    try:
        r = session.get(f"{LRCLIB_BASE}/search",
                        params={"track_name": t, "artist_name": a}, timeout=15)
        if r.status_code == 200 and r.json():
            return r.json()[0]
    except requests.RequestException:
        pass
    return None

# ---------- API 2: lyrics.ovh ----------
def busca_lyricsovh(titulo: str, artista: str) -> str | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        r = session.get(f"{LYRICSOVH_BASE}/{a}/{t}", timeout=15)
        if r.status_code == 200:
            return r.json().get("lyrics")
    except requests.RequestException:
        pass
    return None

# ---------- API 3: Genius ----------
# Passa o token explicitamente (nem toda versão da lib lê a env var sozinha)
genius = Genius(os.environ["GENIUS_ACCESS_TOKEN"])
genius.remove_section_headers = True
genius.skip_non_songs = True
genius.excluded_terms = ["(Remix)", "(Live)", "(Acoustic)", "Ao Vivo", "(Ao Vivo)"]

def busca_genius(titulo: str, artista: str) -> str | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        song = genius.search_song(t, a)
        if song and song.lyrics:
            return song.lyrics
    except Exception as e:
        print(f"Erro Genius '{t}' - '{a}': {e}")
    finally:
        time.sleep(1.0)
    return None

# ---------- BUSCA UNIFICADA ----------
def busca_todas_fontes(titulo: str, artista: str, duracao=None) -> dict:
    resultado = {
        "letra_lrclib":  None,
        "letra_ovh":     None,
        "letra_genius":  None,
        "instrumental":  None,
    }
    res = busca_lrclib(titulo, artista, duracao)
    if res and res.get("plainLyrics"):
        resultado["letra_lrclib"] = res["plainLyrics"]
        resultado["instrumental"] = res.get("instrumental")

    letra_ovh = busca_lyricsovh(titulo, artista)
    if letra_ovh:
        resultado["letra_ovh"] = letra_ovh

    if not resultado["letra_lrclib"] and not resultado["letra_ovh"]:
        letra_genius = busca_genius(titulo, artista)
        if letra_genius:
            resultado["letra_genius"] = letra_genius

    return resultado

# =========================================================
# ---------- LIMPEZA DO CACHE (remove "sem resultado") ----------
# =========================================================
def _tem_texto(x) -> bool:
    return isinstance(x, str) and x.strip() != ""

def limpa_cache_vazios(cache: Path, manter_instrumentais: bool = True) -> None:
    """Reescreve o cache mantendo só linhas com letra em >=1 fonte
    (ou instrumentais, se `manter_instrumentais`). Backup + escrita atômica."""
    if not cache.exists() or cache.stat().st_size == 0:
        print(f"[cache] {cache.name} vazio/inexistente — nada a limpar.")
        return

    raw = pd.read_json(cache, lines=True)
    n_antes = len(raw)

    cols_letra = [c for c in ("letra_lrclib", "letra_ovh", "letra_genius") if c in raw.columns]
    mask_ok = raw[cols_letra].apply(lambda s: s.map(_tem_texto)).any(axis=1) if cols_letra \
              else pd.Series(False, index=raw.index)

    if manter_instrumentais and "instrumental" in raw.columns:
        mask_instr = raw["instrumental"].eq(True)   # True→True; False/NaN→False
    else:
        mask_instr = pd.Series(False, index=raw.index)

    mask_manter = mask_ok | mask_instr
    n_remover = int((~mask_manter).sum())

    if n_remover == 0:
        print(f"[cache] já está limpo: {n_antes} entradas, todas com resultado.")
        return

    shutil.copy2(cache, cache.with_suffix(cache.suffix + ".bak"))
    ids_manter = set(raw.loc[mask_manter, "track_id"])

    tmp = cache.with_suffix(cache.suffix + ".tmp")
    with cache.open(encoding="utf-8") as fin, tmp.open("w", encoding="utf-8") as fout:
        for linha in fin:
            try:
                d = json.loads(linha)
            except json.JSONDecodeError:
                continue  # descarta linha corrompida
            if d.get("track_id") in ids_manter:
                fout.write(json.dumps(d, ensure_ascii=False) + "\n")
    tmp.replace(cache)   # atômico

    print(f"[cache] limpo: {n_antes} → {n_antes - n_remover} entradas "
          f"({n_remover} removidas; backup em {cache.name}.bak)")

limpa_cache_vazios(CACHE, manter_instrumentais=True)

# ---------- LOOP COM CACHE ----------
feitos = set()
if CACHE.exists():
    with CACHE.open(encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

cols = ["track_id", "title", "artists"]
if "duration" in top40.columns:
    cols.append("duration")
faixas = top40[cols].drop_duplicates("track_id")

with CACHE.open("a", encoding="utf-8") as f:
    for row in faixas.itertuples():
        if row.track_id in feitos:
            continue

        duracao = getattr(row, "duration", None)
        try:
            res = busca_todas_fontes(row.title, row.artists, duracao)
        except Exception as e:
            print("erro", row.track_id, e)
            continue

        res["track_id"] = row.track_id
        # só grava se pelo menos UMA fonte trouxe letra (ou é instrumental)
        if any(res[k] for k in ("letra_lrclib", "letra_ovh", "letra_genius")) \
           or res.get("instrumental"):
            f.write(json.dumps(res, ensure_ascii=False) + "\n")
            f.flush()
        time.sleep(0.25)

print("Concluído. Total de faixas no cache:", sum(1 for _ in CACHE.open(encoding="utf-8")))

# ---------- RELATÓRIO E ESCOLHA DA MELHOR FONTE ----------
if CACHE.exists() and CACHE.stat().st_size > 0:
    letras = pd.read_json(CACHE, lines=True)
else:
    letras = pd.DataFrame(columns=["track_id", "letra_lrclib", "letra_ovh",
                                   "letra_genius", "instrumental"])

for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
    letras[f"tem_{col}"] = letras[col].map(_tem_texto) if col in letras.columns else False

print("\nCobertura por fonte:")
for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
    print(f"  {col:15s}: {letras[f'tem_{col}'].mean():.1%}")

tem_alguma = letras[["tem_letra_lrclib", "tem_letra_ovh", "tem_letra_genius"]].any(axis=1)
print(f"\nCobertura combinada: {tem_alguma.mean():.1%}")
print(f"Sem letra em nenhuma fonte: {(~tem_alguma).sum()} faixas")

def escolhe_letra(r):
    for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
        if _tem_texto(r.get(col)):
            return r[col], col.replace("letra_", "")
    return None, None

letras[["letra", "fonte"]] = letras.apply(
    lambda r: pd.Series(escolhe_letra(r)), axis=1
)

top40_final1 = top40.merge(
    letras[["track_id", "letra", "fonte", "instrumental"]],
    on="track_id", how="left"
)
print("\nDistribuição da fonte escolhida:")
print(top40_final1["fonte"].value_counts(dropna=False))

Cache em: /home/besome/Documentos/Coding/socialComp/SocialComp/dados/letras_cache.jsonl
[cache] já está limpo: 4961 entradas, todas com resultado.
Concluído. Total de faixas no cache: 4961

Cobertura por fonte:
  letra_lrclib   : 99.5%
  letra_ovh      : 62.6%
  letra_genius   : 0.2%

Cobertura combinada: 100.0%
Sem letra em nenhuma fonte: 0 faixas

Distribuição da fonte escolhida:
fonte
lrclib    9654
ovh         38
NaN         16
genius      12
Name: count, dtype: int64


In [8]:
top40_final1

,cc,year,track_id,pontos,rank_medio,melhor_posicao,semanas,streams,title,artists,letra,fonte,instrumental
0,au,2017,7qiZfU4dY1lWllzX7mPBI3,8967,18.000000,1,49,49345417,Shape of You,Ed Sheeran,The club isn't the best place to find a lover\...,lrclib,0.0
1,au,2017,3kxfsdsCpFgN412fpnW85Y,7472,48.510204,16,49,21999044,Redbone,Childish Gambino,Daylight\nI wake up feeling like you won't pla...,lrclib,0.0
2,au,2017,3a1lNhkSLSkpJE4MSHpDu9,7139,38.750000,15,44,25151833,Congratulations,"Post Malone, Quavo","Yeah, yeah\nYeah, yeah\nYeah, yeah\nYeah (hey)...",lrclib,0.0
3,au,2017,6PCUP3dWmTjcTtXY02oFdT,7110,55.897959,2,49,25704249,Castle on the Hill,Ed Sheeran,"When I was six years old, I broke my leg\n\nI ...",lrclib,0.0
4,au,2017,5uCax9HTNlzGybIStD3vDh,6939,59.387755,9,49,17786120,Say You Won't Let Go,James Arthur,I met you in the dark\nYou lit me up\nYou made...,lrclib,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9715,us,2025,43iIQbw5hx986dUEZbr3eN,668,126.777778,69,9,23767403,From The Start,Laufey,Don't you notice how\nI get quiet when there's...,lrclib,0.0
9716,us,2025,0l0vcZMU7AOeQmUIREoI2U,667,117.625000,43,8,23382204,Qué Pasaría...,Rauw Alejandro | Bad Bunny,¿Qué pasaría si volvieran esas noche' dentro d...,lrclib,0.0
9717,us,2025,4eLDmhsJW3JoZTXCAozHor,654,92.000000,42,6,20646073,Loco,Neton Vega,Hey xolo yo no me equivoco\nSi monto un Reguet...,lrclib,0.0
9718,us,2025,1UHS8Rf6h5Ar3CDWRd3wjF,653,92.166667,45,6,18896758,Bed Chem,Sabrina Carpenter,I was in a sheer dress the day that we met\nWe...,lrclib,0.0


In [9]:
sem = top40_final1[top40_final1["fonte"].isna()]
print(len(sem), "linhas |", sem["track_id"].nunique(), "tracks únicas")

16 linhas | 13 tracks únicas


In [10]:
# faltantes = top40_final1[top40_final1["fonte"].isna()]

# # se quiser separar as instrumentais (não precisam de letra):
# if "instrumental" in top40_final1.columns:
#     faltantes = top40_final1[(top40_final1["fonte"].isna() ) & (top40_final1["instrumental"] != True)]

# faltantes = faltantes.drop_duplicates("track_id")[["track_id", "title", "artists", "cc"]]

In [11]:
# faltantes.count()

In [12]:
# ---------- configuração ----------
COL = "letra"                       # coluna de letra em top40_final1
CACHE = "letras_fallback3.jsonl"    # cache novo (só guarda sucessos)
RESET_CACHE = False                 # True apaga o cache e refaz tudo
PAUSA = 0.4
VAGALUME_KEY = None                 # opcional; None = não usa

session = requests.Session()
session.headers.update({"User-Agent": "top40_final1-letras/0.1 (seu-email-ou-repo)"})

# ---------- limpeza e variantes de busca ----------

def lista_artistas(a):
    return [x.strip() for x in re.split(r"\s*\|\s*", a) if x.strip()]   # não divide em "&" nem ","

def variantes(titulo, artistas):
    t = limpa_titulo(titulo)
    arts = lista_artistas(artistas) or [""]
    for a in arts[:2]:
        yield "artista", t, a
        if "&" in a:
            yield "artista_e", t, a.replace("&", "e")   # "Matheus e Kauan"
    yield "titulo_original", titulo.split(" - ")[0].strip(), arts[0]
    yield "so_titulo", t, ""                            # menos confiável, por último

# ---------- fontes: cada uma devolve (letra, titulo_achado, artista_achado) ou None ----------
def busca_lrclib2(t, a):
    r = session.get("https://lrclib.net/api/search", params={"q": f"{t} {a}".strip()}, timeout=15)
    r.raise_for_status()
    for item in r.json():
        if item.get("plainLyrics"):
            return item["plainLyrics"], item.get("trackName"), item.get("artistName")
    return None

def busca_syncedlyrics(t, a):
    letra = syncedlyrics.search(f"{t} {a}".strip(), plain_only=True)
    return (letra, None, None) if letra else None

def busca_vagalume(t, a):
    if not a or not VAGALUME_KEY:
        return None
    r = session.get("https://api.vagalume.com.br/search.php",
                    params={"art": a, "mus": t, "apikey": VAGALUME_KEY}, timeout=15)
    r.raise_for_status()
    j = r.json()
    if j.get("type") in ("exact", "aprox") and j.get("mus"):
        return j["mus"][0].get("text"), j["mus"][0].get("name"), a
    return None

fontes = [("lrclib2", busca_lrclib2), ("syncedlyrics", busca_syncedlyrics)]
if VAGALUME_KEY:
    fontes.append(("vagalume", busca_vagalume))

# ---------- 1. limpeza do top40_final1 ----------
sobras = [c for c in top40_final1.columns
          if c.endswith(("_x", "_y")) or c in ["Unnamed: 0", "letra_fb", "fonte_fb", "variante",
                                                "match_titulo", "match_artista"]]
top40_final1 = top40_final1.drop(columns=sobras)
top40_final1[COL] = top40_final1[COL].fillna("").astype(str)
if "fonte" not in top40_final1.columns:
    top40_final1["fonte"] = np.nan

# ---------- 2. faltantes únicas (ignora instrumentais) ----------
faltantes = top40_final1[top40_final1["fonte"].isna()]
if "instrumental" in top40_final1.columns:
    faltantes = faltantes[faltantes["instrumental"].ne(True)]
faltantes = faltantes.drop_duplicates("track_id")[["track_id", "title", "artists", "cc"]]
print(len(faltantes), "tracks únicas sem letra")

# ---------- 3. loop com cache (só grava sucessos) ----------
if RESET_CACHE and os.path.exists(CACHE):
    os.remove(CACHE)

feitos = set()
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

with open(CACHE, "a", encoding="utf-8") as f:
    for row in faltantes.itertuples():
        if row.track_id in feitos:
            continue
        achou = None
        for nome, fn in fontes:
            for tipo, t, a in variantes(row.title, row.artists):
                try:
                    res = fn(t, a)
                except Exception as e:
                    print("erro", nome, row.track_id, repr(e)[:120])
                    res = None
                time.sleep(PAUSA)
                if res and res[0]:
                    achou = {"track_id": row.track_id, "letra_fb": res[0], "fonte_fb": nome,
                             "variante": tipo, "match_titulo": res[1], "match_artista": res[2]}
                    break
            if achou:
                break
        if achou:
            f.write(json.dumps(achou, ensure_ascii=False) + "\n")
            f.flush()

# ---------- 4. merge de volta em top40_final1 (só as colunas necessárias) ----------
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    fb = pd.read_json(CACHE, lines=True).drop_duplicates("track_id")
    print("achadas por fonte/variante:")
    print(fb.groupby(["fonte_fb", "variante"]).size())

    mapa_letra = fb.set_index("track_id")["letra_fb"]
    mapa_fonte = fb.set_index("track_id")["fonte_fb"]

    novas = top40_final1["track_id"].map(mapa_letra)
    preencher = (top40_final1[COL].str.strip() == "") & novas.notna()
    top40_final1.loc[preencher, COL] = novas[preencher]
    top40_final1.loc[preencher, "fonte"] = top40_final1.loc[preencher, "track_id"].map(mapa_fonte)
else:
    fb = pd.DataFrame()
    print("o fallback não achou nada novo")

if "encontrada" in top40_final1.columns:
    top40_final1["encontrada"] = top40_final1[COL].str.strip() != ""

# ---------- 5. resultado e revisão ----------
ainda = top40_final1[top40_final1[COL].str.strip() == ""].drop_duplicates("track_id")
print(len(ainda), "tracks únicas ainda sem letra")
print(ainda["cc"].value_counts())
display(ainda[["cc", "title", "artists"]])

# conferir se a música achada é a certa (principalmente "so_titulo")
if not fb.empty:
    rev = fb.merge(top40_final1[["track_id", "cc", "title", "artists"]].drop_duplicates("track_id"), on="track_id")
    rev["inicio"] = rev["letra_fb"].str[:80].str.replace("\n", " / ")
    display(rev[["cc", "title", "artists", "match_titulo", "match_artista",
                 "fonte_fb", "variante", "inicio"]].sort_values("variante"))

13 tracks únicas sem letra
achadas por fonte/variante:
fonte_fb  variante       
lrclib2   artista            5
          so_titulo          6
          titulo_original    1
dtype: int64
1 tracks únicas ainda sem letra
cc
gb    1
Name: count, dtype: int64


,cc,title,artists
6285,gb,Clean Baby Sleep White Noise (Loopable),Dream Supplier


,cc,title,artists,match_titulo,match_artista,fonte_fb,variante,inicio
0,br,sentaDONA (Remix) s2,Davi Kneip | Mc Frog | Dj Gabriel do Borel | L...,sentaDONA (Remix) s2,Davi Kneip,lrclib2,artista,"MC-MC Frog / É Davi Kneip, tá bom? / Gabriel d..."
3,jp,ぎゅっと。,もさを。,ぎゅっと。,もさを。,lrclib2,artista,手を繋ぐと体温が上がって / 無邪気な笑顔につられて / 変な口癖真似してみたり / 寝顔を...
5,jp,青のすみか,Tatsuya Kitani,青のすみか / キタニタツヤ - Where Our Blue Is / Tatsuya K...,キタニタツヤ / Tatsuya Kitani,lrclib2,artista,どこまでも続くような青の季節は / 四つ並ぶ眼の前を遮るものは何もない / アスファルト、蝉...
7,jp,君のまま,MUKADE | INMAN,百足&韻マン - 君のまま,韻マン (inman),lrclib2,artista,"Ah, yeah (Runaway, Kontrabandz) / Mhm, yeah ..."
8,jp,青のすみか,Tatsuya Kitani,青のすみか / キタニタツヤ - Where Our Blue Is / Tatsuya K...,キタニタツヤ / Tatsuya Kitani,lrclib2,artista,どこまでも続くような青の季節は / 四つ並ぶ眼の前を遮るものは何もない / アスファルト、蝉...
1,br,Como Um Beijo,amblu,Como Um Beijo,"Marcos & Belutti,Nadson O Ferinha",lrclib2,so_titulo,"A noite em que te conheci / Era só você e eu, ..."
4,jp,レオ,Yuuri,レオ,優里,lrclib2,so_titulo,ショーケースの中過ごしていた / 誰もかれもが過ぎ去っていた / 怖かったんだ あの日君に ...
6,jp,"Hontowane,",yangskinny,Hontowane,Aqua Timez,lrclib2,so_titulo,飼ってた犬も 生まれた土地も / こっそり抱えてきた悩み事も / 夢中になった部活も 土の匂...
9,jp,Love Song,Marcy,A Love Song,Love Song,lrclib2,so_titulo,Lend an ear to a love song / Oooh a love song ...
10,jp,My lovely killer,noa,初恋キラー,乃紫,lrclib2,so_titulo,"イジワルは少年の I love you (I love you, love you) / 泣..."


In [13]:
# Filtra mantendo apenas músicas que NÃO são instrumentais E que possuem letra com 20 ou mais caracteres
condicao_manter = (top40_final1["instrumental"] != True) & (top40_final1["letra"].fillna("").astype(str).str.len() >= 20)

# Aplica o filtro
top40_final1 = top40_final1[condicao_manter]

# Exibe o novo tamanho e uma amostra
print(f"Filtrado! Novo tamanho: {top40_final1.shape[0]} linhas")
display(top40_final1.head())

Filtrado! Novo tamanho: 9717 linhas


,cc,year,track_id,pontos,rank_medio,melhor_posicao,semanas,streams,title,artists,letra,fonte,instrumental
0,au,2017,7qiZfU4dY1lWllzX7mPBI3,8967,18.000000,1,49,49345417,Shape of You,Ed Sheeran,The club isn't the best place to find a lover\...,lrclib,0.0
1,au,2017,3kxfsdsCpFgN412fpnW85Y,7472,48.510204,16,49,21999044,Redbone,Childish Gambino,Daylight\nI wake up feeling like you won't pla...,lrclib,0.0
2,au,2017,3a1lNhkSLSkpJE4MSHpDu9,7139,38.750000,15,44,25151833,Congratulations,"Post Malone, Quavo","Yeah, yeah\nYeah, yeah\nYeah, yeah\nYeah (hey)...",lrclib,0.0
3,au,2017,6PCUP3dWmTjcTtXY02oFdT,7110,55.897959,2,49,25704249,Castle on the Hill,Ed Sheeran,"When I was six years old, I broke my leg\n\nI ...",lrclib,0.0
4,au,2017,5uCax9HTNlzGybIStD3vDh,6939,59.387755,9,49,17786120,Say You Won't Let Go,James Arthur,I met you in the dark\nYou lit me up\nYou made...,lrclib,0.0


In [14]:
saveArchiveB(top40_final1, "./completoComLetras")

Salvo em ./completoComLetras.csv | 9717 linhas x 13 colunas
Forma preservada: True | Letras idênticas: True | Letras truncadas: 0


'./completoComLetras.csv'